# CSR Array – PYNQ Interactive Test

This notebook exercises the **csr_array** AXI4-Lite register block on a live PYNQ board.

| Offset | Name | Access | Description |
|--------|------|--------|-------------|
| 0x00 | CTRL | R/W | bit0: START, bit1: SOFT_RESET, bit2: IRQ_EN |
| 0x04 | STATUS | R | bit0: BUSY, bit1: DONE, bit2: ERROR |
| 0x08 | CFG | R/W | bit0: CAUSAL_EN |
| 0x14–0x18 | Q_BASE | R/W | Q base address (64-bit, split) |
| 0x1C–0x20 | K_BASE | R/W | K base address (64-bit, split) |
| 0x24–0x28 | V_BASE | R/W | V base address (64-bit, split) |
| 0x2C–0x30 | O_BASE | R/W | O base address (64-bit, split) |
| 0x34 | STRIDE_BYTES | R/W | Row stride |
| 0x38 | NEG_LARGE | R/W | -inf approximation |
| 0x3C | SCALE | R/W | 1/sqrt(d) |
| 0x40 | CYCLES | R | Execution cycle count |

## 1. Load Overlay

In [5]:
from pynq import Overlay, MMIO

BITSTREAM = "/home/xilinx/jupyter_notebooks/syc/flash_atten/csr_lab/design_1.bit"   # <-- update path
BASE_ADDR = 0xA000_0000                           # <-- update if different
ADDR_RANGE = 0x44

ol = Overlay(BITSTREAM)
mmio = MMIO(BASE_ADDR, ADDR_RANGE)
print("Overlay loaded, MMIO ready.")

try:
    csr_ip = ol.csr_array_0
    print(f"Successfully located IP: {csr_ip.mmio.base_addr:08X}")
except AttributeError:
    print("ERROR: Could not find csr_array_0 in the design. Check your Block Design")
    sys.exit(1)




Overlay loaded, MMIO ready.
Successfully located IP: A0000000


## 2. Register Map Helpers

In [6]:
REG = {
    "CTRL":         0x00,
    "STATUS":       0x04,
    "CFG":          0x08,
    "Q_BASE_L":     0x14,
    "Q_BASE_H":     0x18,
    "K_BASE_L":     0x1C,
    "K_BASE_H":     0x20,
    "V_BASE_L":     0x24,
    "V_BASE_H":     0x28,
    "O_BASE_L":     0x2C,
    "O_BASE_H":     0x30,
    "STRIDE_BYTES": 0x34,
    "NEG_LARGE":    0x38,
    "SCALE":        0x3C,
    "CYCLES":       0x40,
}

def wr(name, val):  mmio.write(REG[name], val)
def rd(name):       return mmio.read(REG[name])

def dump():
    for name in sorted(REG, key=REG.get):
        print(f"  0x{REG[name]:02X}  {name:16s} = 0x{rd(name):08X}")

## 3. Test – Reset Values
All R/W registers should read 0 after loading the overlay.

In [7]:
RW_REGS = [
    "CTRL", "CFG",
    "Q_BASE_L", "Q_BASE_H", "K_BASE_L", "K_BASE_H",
    "V_BASE_L", "V_BASE_H", "O_BASE_L", "O_BASE_H",
    "STRIDE_BYTES", "NEG_LARGE", "SCALE",
]

for name in RW_REGS:
    val = rd(name)
    status = "PASS" if val == 0 else "FAIL"
    print(f"  [{status}] {name} = 0x{val:08X}")

  [PASS] CTRL = 0x00000000
  [PASS] CFG = 0x00000000
  [PASS] Q_BASE_L = 0x00000000
  [PASS] Q_BASE_H = 0x00000000
  [PASS] K_BASE_L = 0x00000000
  [PASS] K_BASE_H = 0x00000000
  [PASS] V_BASE_L = 0x00000000
  [PASS] V_BASE_H = 0x00000000
  [PASS] O_BASE_L = 0x00000000
  [PASS] O_BASE_H = 0x00000000
  [PASS] STRIDE_BYTES = 0x00000000
  [PASS] NEG_LARGE = 0x00000000
  [PASS] SCALE = 0x00000000


## 4. Test – Write / Read-back

In [8]:
patterns = [0xDEAD_BEEF, 0x1234_5678, 0x0000_0001, 0xFFFF_FFFF, 0x0000_0000]
fails = 0

for pat in patterns:
    for name in RW_REGS:
        wr(name, pat)
    for name in RW_REGS:
        val = rd(name)
        if val != pat:
            print(f"  [FAIL] {name}: wrote 0x{pat:08X}, read 0x{val:08X}")
            fails += 1

print(f"Write/Read-back: {'PASS' if fails == 0 else f'{fails} FAILURES'}")

Write/Read-back: PASS


## 5. Test – CTRL Bit-fields

In [9]:
for bit_name, bit_val in [("START", 1), ("SOFT_RESET", 2), ("IRQ_EN", 4)]:
    wr("CTRL", bit_val)
    val = rd("CTRL")
    status = "PASS" if val == bit_val else "FAIL"
    print(f"  [{status}] CTRL.{bit_name}: 0x{val:08X}")

wr("CTRL", 0x7)
val = rd("CTRL")
print(f"  [{'PASS' if val == 7 else 'FAIL'}] CTRL all bits: 0x{val:08X}")

wr("CTRL", 0)  # clean up

  [PASS] CTRL.START: 0x00000001
  [PASS] CTRL.SOFT_RESET: 0x00000002
  [PASS] CTRL.IRQ_EN: 0x00000004
  [PASS] CTRL all bits: 0x00000007


## 6. Test – 64-bit Addresses

In [10]:
bases = {
    "Q_BASE": (0xDEAD_BEEF, 0x0102_0304),
    "K_BASE": (0x1111_2222, 0x3333_4444),
    "V_BASE": (0xAAAA_BBBB, 0xCCCC_DDDD),
    "O_BASE": (0x5555_6666, 0x7777_8888),
}

for name, (lo, hi) in bases.items():
    wr(f"{name}_L", lo)
    wr(f"{name}_H", hi)

for name, (lo, hi) in bases.items():
    rd_l = rd(f"{name}_L")
    rd_h = rd(f"{name}_H")
    got = (rd_h << 32) | rd_l
    exp = (hi  << 32) | lo
    status = "PASS" if got == exp else "FAIL"
    print(f"  [{status}] {name}: 0x{got:016X}")

  [PASS] Q_BASE: 0x01020304DEADBEEF
  [PASS] K_BASE: 0x3333444411112222
  [PASS] V_BASE: 0xCCCCDDDDAAAABBBB
  [PASS] O_BASE: 0x7777888855556666


## 7. Test – STATUS is Read-Only

In [11]:
before = rd("STATUS")
wr("STATUS", 0xFFFF_FFFF)
after = rd("STATUS")
status = "PASS" if before == after else "FAIL"
print(f"  [{status}] STATUS unchanged: 0x{after:08X}")

  [PASS] STATUS unchanged: 0x00000000


## 8. Register Dump

In [12]:
dump()

  0x00  CTRL             = 0x00000000
  0x04  STATUS           = 0x00000000
  0x08  CFG              = 0x00000000
  0x14  Q_BASE_L         = 0xDEADBEEF
  0x18  Q_BASE_H         = 0x01020304
  0x1C  K_BASE_L         = 0x11112222
  0x20  K_BASE_H         = 0x33334444
  0x24  V_BASE_L         = 0xAAAABBBB
  0x28  V_BASE_H         = 0xCCCCDDDD
  0x2C  O_BASE_L         = 0x55556666
  0x30  O_BASE_H         = 0x77778888
  0x34  STRIDE_BYTES     = 0x00000000
  0x38  NEG_LARGE        = 0x00000000
  0x3C  SCALE            = 0x00000000
  0x40  CYCLES           = 0x00000000
